<h1 align="center">🖥️ ColabDesk</h1>
<p align="center"><strong>A Minimalist Google Colab Virtual Desktop (XFCE4) via Browser</strong></p>

---
### ⚡ Features
- **One-Click Run**: Everything is integrated into a single execution cell.
- **No App Needed**: Access the desktop directly from your web browser (noVNC).
- **Smart Token Fallback**: Paste as many Ngrok tokens as you want. The script automatically shuffles and tries them until one succeeds.

In [ ]:
# @title 🚀 Start ColabDesk (One-Click Execution)
# @markdown Fill in the options below and click the **Play** button on the left.
# @markdown ---
vnc_password = "colab123" # @param {type:"string"}
resolution = "1280x720" # @param ["1024x768", "1280x720", "1920x1080"]

# @markdown **Ngrok Tokens:** Paste all your tokens here (separate multiple tokens with a comma ","). 
# @markdown *(Note: The system will automatically separate them whether you use commas, spaces, or newlines).* 
ngrok_tokens = "" # @param {type:"string"}
# @markdown ---

import os
import time
import json
import urllib.request
import random
import re
from IPython.display import clear_output

# Configuration to prevent apt install hanging
os.environ["DEBIAN_FRONTEND"] = "noninteractive"

print("⏳ 1/4: Cleaning up previous sessions...")
os.system("vncserver -kill :1 > /dev/null 2>&1")
os.system("rm -rf /tmp/.X1* /tmp/.X11-unix")
os.system("pkill -f websockify")
os.system("pkill -f ngrok")

print("⏳ 2/4: Installing XFCE4 Desktop & Dependencies (Takes ~2 minutes, please wait)...")
# Install quietly but output to log file in case of debugging needs
os.system("apt-get update -qq > /tmp/apt.log 2>&1")
os.system("apt-get install -y -qq xfce4 xfce4-goodies tightvncserver novnc websockify >> /tmp/apt.log 2>&1")
print("⏳ Installing Google Chrome...")
os.system("wget -q -O chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
os.system("apt-get install -y -qq ./chrome.deb >> /tmp/apt.log 2>&1")
os.system("rm chrome.deb")
os.system("sed -i 's/google-chrome-stable/google-chrome-stable --no-sandbox --disable-dev-shm-usage/g' /usr/share/applications/google-chrome.desktop")
os.system("mkdir -p ~/.config/autostart")
os.system("cp /usr/share/applications/google-chrome.desktop ~/.config/autostart/")

print("⏳ 3/4: Downloading & Configuring Ngrok and VNC...")
os.system("rm -f /usr/local/bin/ngrok")
os.system("wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz")
os.system("tar -xzf ngrok-v3-stable-linux-amd64.tgz -C /usr/local/bin > /dev/null 2>&1")
os.system("rm ngrok-v3-stable-linux-amd64.tgz")

os.system("mkdir -p ~/.vnc")
os.system(f"echo '{vnc_password}' | vncpasswd -f > ~/.vnc/passwd")
os.system("chmod 600 ~/.vnc/passwd")

xstartup = """#!/bin/bash
xrdb $HOME/.Xresources
startxfce4 &
"""
with open("/root/.vnc/xstartup", "w") as f:
    f.write(xstartup)
os.system("chmod +x /root/.vnc/xstartup")

print(f"⏳ 4/4: Starting VNC Server ({resolution}) & noVNC...")
os.system(f"USER=root vncserver -geometry {resolution} :1 > /dev/null 2>&1")
os.system("websockify --web /usr/share/novnc 6080 localhost:5901 > /dev/null 2>&1 &")

# Smart Token Parsing
tokens = [t.strip() for t in re.split(r'[\s,]+', ngrok_tokens) if t.strip()]

if not tokens:
    clear_output()
    print("❌ ERROR: You must provide at least one Ngrok token in the 'ngrok_tokens' field!")
else:
    random.shuffle(tokens)
    ngrok_url = None

    for token in tokens:
        print(f"\n🔄 Connecting to Ngrok with token: {token[:4]}...{token[-4:]}")
        os.system(f"ngrok config add-authtoken {token} > /dev/null 2>&1")
        os.system("ngrok http 6080 --log=stdout > /tmp/ngrok.log 2>&1 &")
        
        time.sleep(5) # Give ngrok time to establish the tunnel
        
        try:
            req = urllib.request.Request("http://localhost:4040/api/tunnels")
            with urllib.request.urlopen(req) as response:
                data = json.loads(response.read().decode())
                if data.get('tunnels'):
                    ngrok_url = data['tunnels'][0]['public_url']
                    break
        except Exception as e:
            print(f"⚠️ Rate-limited or invalid token. Error: {e}")
            os.system("cat /tmp/ngrok.log | tail -n 5")
            os.system("pkill -f ngrok")

    clear_output()
    
    if ngrok_url:
        print("🎉 COLABDESK IS READY! 🎉")
        print("="*60)
        print(f"🔗 CLICK HERE TO OPEN: {ngrok_url}/vnc.html")
        print("="*60)
        print(f"👉 VNC Password: {vnc_password}")
        print("\n⚠️ IMPORTANT: Keep this Colab tab open in your browser so the session doesn't disconnect.")
    else:
        print("❌ FAILED: All provided Ngrok tokens are invalid or rate-limited.")
        print("Please provide fresh tokens from https://dashboard.ngrok.com")